# Chapitre 2 — Extraction des données (sources locales)


---

## Objectifs d'apprentissage

À la fin de ce chapitre, vous serez capable de :

1. **Extraire** des données depuis différents formats de fichiers (CSV, Excel, JSON) en utilisant les paramètres appropriés de pandas
2. **Connecter** Python à une base de données SQL et exécuter des requêtes pour récupérer des données dans un DataFrame
3. **Consommer** des APIs REST en gérant l'authentification, la pagination et les limites de requêtes
4. **Évaluer** quand le web scraping est approprié et appliquer les principes éthiques et légaux

---

## 2.5 Inspection initiale des données

### Les premiers réflexes

Après avoir extrait vos données, **ne vous lancez jamais directement dans l'analyse**. D'abord, inspectez.

In [ ]:
# Les 5 commandes essentielles
# df.shape        # Dimensions (lignes, colonnes)
# df.dtypes       # Types de données
# df.head()       # Premiers enregistrements
# df.info()       # Résumé complet
# df.describe()   # Statistiques numériques

### Exemple complet d'inspection

In [63]:
import pandas as pd
import numpy as np

# Créons des données d'exemple
np.random.seed(42)
df_ventes = pd.DataFrame({
    'date': pd.date_range('2024-01-01', periods=100, freq='D'),
    'produit': np.random.choice(['A', 'B', 'C'], 100),
    'quantite': np.random.randint(1, 50, 100),
    'prix_unitaire': np.random.uniform(10, 100, 100).round(2),
    'region': np.random.choice(['Nord', 'Sud', 'Est', 'Ouest', None], 100)  # Avec valeurs manquantes
})

# 1. Dimensions
print(f"📊 Dimensions : {df_ventes.shape[0]} lignes × {df_ventes.shape[1]} colonnes")

📊 Dimensions : 100 lignes × 5 colonnes


In [64]:
# 2. Types de données
print("📝 Types de données :")
print(df_ventes.dtypes)

📝 Types de données :
date             datetime64[ns]
produit                  object
quantite                  int64
prix_unitaire           float64
region                   object
dtype: object


In [65]:
# 3. Aperçu
print("👀 Aperçu des données :")
df_ventes.head()

👀 Aperçu des données :


,date,produit,quantite,prix_unitaire,region
0,2024-01-01,C,44,60.23,Ouest
1,2024-01-02,A,8,46.35,Est
2,2024-01-03,C,24,15.84,Nord
3,2024-01-04,C,11,32.85,Nord
4,2024-01-05,A,17,32.22,Ouest


In [66]:
# 4. Résumé mémoire
print("💾 Informations mémoire :")
df_ventes.info()

💾 Informations mémoire :
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   date           100 non-null    datetime64[ns]
 1   produit        100 non-null    object        
 2   quantite       100 non-null    int64         
 3   prix_unitaire  100 non-null    float64       
 4   region         81 non-null     object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(2)
memory usage: 4.0+ KB


In [67]:
# 5. Statistiques numériques
print("📈 Statistiques :")
df_ventes.describe()

📈 Statistiques :


,date,quantite,prix_unitaire
count,100,100.000000,100.000000
mean,2024-02-19 12:00:00,25.900000,59.270900
min,2024-01-01 00:00:00,1.000000,11.630000
25%,2024-01-25 18:00:00,13.000000,38.937500
50%,2024-02-19 12:00:00,28.000000,58.255000
75%,2024-03-15 06:00:00,37.000000,80.972500
max,2024-04-09 00:00:00,49.000000,99.800000
std,NaN,13.826851,24.661466


---

### Checklist d'inspection initiale

| Vérification | Commande | Question |
|--------------|----------|---------|
| Nombre de lignes | `df.shape[0]` | Est-ce cohérent avec la source ? |
| Nombre de colonnes | `df.shape[1]` | Manque-t-il des colonnes ? |
| Types de données | `df.dtypes` | Les dates sont-elles reconnues ? |
| Valeurs nulles | `df.isnull().sum()` | Combien de données manquantes ? |
| Doublons | `df.duplicated().sum()` | Y a-t-il des lignes en double ? |
| Valeurs uniques | `df["col"].nunique()` | Combien de catégories ? |

In [ ]:
# Application de la checklist
print("📋 CHECKLIST D'INSPECTION")
print("=" * 40)
print(f"Lignes : {df_ventes.shape[0]}")
print(f"Colonnes : {df_ventes.shape[1]}")
print(f"\nValeurs nulles par colonne :")
print(df_ventes.isnull().sum())
print(f"\nDoublons : {df_ventes.duplicated().sum()}")
print(f"\nValeurs uniques 'produit' : {df_ventes['produit'].nunique()}")
print(f"Valeurs uniques 'region' : {df_ventes['region'].nunique()}")

---

### Documentation de la provenance (Data Lineage)

**Toujours documenter d'où viennent vos données !**

In [68]:
import json
from datetime import datetime

# Exemple de métadonnées à conserver
metadata = {
    "source": "Données simulées pour exercice",
    "date_extraction": datetime.now().isoformat(),
    "nb_enregistrements": len(df_ventes),
    "colonnes": list(df_ventes.columns),
    "filtres_appliques": "Aucun",
    "responsable": "etudiant@formation.com"
}

print("📄 Métadonnées du dataset :")
print(json.dumps(metadata, indent=2, ensure_ascii=False))

# Sauvegarder avec les données (optionnel)
# with open("ventes_metadata.json", "w") as f:
#     json.dump(metadata, f, indent=2, ensure_ascii=False)

📄 Métadonnées du dataset :
{
  "source": "Données simulées pour exercice",
  "date_extraction": "2026-01-16T16:13:27.912982",
  "nb_enregistrements": 100,
  "colonnes": [
    "date",
    "produit",
    "quantite",
    "prix_unitaire",
    "region"
  ],
  "filtres_appliques": "Aucun",
  "responsable": "etudiant@formation.com"
}


---

### ✍️ Exercice 2.7 : Inspection complète (15 min)

Vous recevez un fichier `clients.csv`. Réalisez l'inspection complète et répondez aux questions.

In [69]:
import pandas as pd

# Simulation de données avec problèmes
df_clients = pd.DataFrame({
    'client_id': [1, 2, 3, 4, 5, 5],  # Doublon !
    'nom': ['Alice', 'Bob', 'Charlie', None, 'Eve', 'Eve'],
    'email': ['alice@test.com', 'bob@test.com', None, 'david@test.com', 'eve@test.com', 'eve@test.com'],
    'age': [25, 30, 'trente-cinq', 40, 28, 28],  # Erreur de type !
    'date_inscription': ['2024-01-15', '2024-02-20', '2024-03-10', '2024-04-05', '2024-05-12', '2024-05-12']
})

print("Données à inspecter :")
df_clients

Données à inspecter :


,client_id,nom,email,age,date_inscription
0,1,Alice,alice@test.com,25,2024-01-15
1,2,Bob,bob@test.com,30,2024-02-20
2,3,Charlie,None,trente-cinq,2024-03-10
3,4,None,david@test.com,40,2024-04-05
4,5,Eve,eve@test.com,28,2024-05-12
5,5,Eve,eve@test.com,28,2024-05-12


In [70]:
# Votre inspection - complétez les blancs
print("1. Dimensions :", df_clients.shape)
print("\n2. Types :")
print(df_clients.dtypes)
print("\n3. Valeurs nulles :")
print(df_clients.isnull().sum())
print("\n4. Doublons :", df_clients.duplicated().sum())

1. Dimensions : (6, 5)

2. Types :
client_id            int64
nom                 object
email               object
age                 object
date_inscription    object
dtype: object

3. Valeurs nulles :
client_id           0
nom                 1
email               1
age                 0
date_inscription    0
dtype: int64

4. Doublons : 1


**Questions :**
- a) Combien de valeurs manquantes au total ?
- b) Quel problème voyez-vous dans la colonne 'age' ?
- c) Combien de doublons exacts y a-t-il ?